# Project 2: Data Preprocessing Pipeline

**Dataset:** California Housing

**Goal:** Master the complete data preprocessing workflow — handling missing values, outliers, feature selection, scaling, and dimensionality reduction.

In [ ]:
# ====== Setup ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    df = pd.read_csv(f'{DATA_URL}/{fn}')
    print(f'Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols')
    return df
print('Setup OK!')
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

## 1. Data Quality — Real-World Problems

In [ ]:
# Load data and inject realistic problems
df = load_data('housing.csv')
np.random.seed(42)
df_dirty = df.copy()

# Problem 1: Missing values — 缺失值 (5% randomly removed)
for col in ['AveRooms', 'AveBedrms']:
    mask = np.random.random(len(df_dirty)) < 0.05
    df_dirty.loc[mask, col] = np.nan
print('Missing values injected in AveRooms and AveBedrms')

# Problem 2: Outliers — 异常值 (unrealistically large room counts)
outlier_idx = np.random.choice(len(df_dirty), 20, replace=False)
max_rooms = df['AveRooms'].max()
df_dirty.loc[outlier_idx, 'AveRooms'] = max_rooms * np.random.uniform(3, 8, 20)
print(f'20 outlier values injected in AveRooms (up to {max_rooms*8:.1f} rooms)')

# Problem 3: Impossible value — 不可能的值 (negative bedrooms!)
df_dirty.loc[np.random.randint(0, len(df_dirty)), 'AveBedrms'] = -1
print('1 impossible value (AveBedrms = -1) injected')

print('\nMissing value report:')
print(df_dirty.isnull().sum().to_string())

## 2. Outlier Detection — IQR Method

In [ ]:
# IQR: Q1 - 1.5*IQR to Q3 + 1.5*IQR
num_cols = ['MedInc','HouseAge','AveRooms','AveBedrms',
            'Population','AveOccup','MedHouseVal']
print(f'{"Column":15s} {"Q1":8s} {"Q3":8s} {"IQR":8s} {"Outliers":>8s}')
print('-' * 50)
for col in num_cols:
    Q1 = df_dirty[col].quantile(0.25)
    Q3 = df_dirty[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR; upper = Q3 + 1.5*IQR
    n_out = ((df_dirty[col] < lower) | (df_dirty[col] > upper)).sum()
    print(f'{col:15s} {Q1:8.3f} {Q3:8.3f} {IQR:8.3f} {n_out:8d}')

## 3. Feature Selection — Three Strategies

In [ ]:
# Strategy 1: Correlation Filter — 过滤法
corr = df.select_dtypes(include=[np.number]).corr()['MedHouseVal'].abs().sort_values()
print('Filter Method — Absolute correlation with target:')
print(corr.to_string())

# Strategy 2: SelectKBest — F-regression
X = df[num_cols[:-1]]; y = df['MedHouseVal']
selector = SelectKBest(score_func=f_regression, k='all').fit(X, y)
scores = pd.DataFrame({'Feature': X.columns, 'F_Score': selector.scores_})
print('\nSelectKBest F-Scores:')
print(scores.sort_values('F_Score', ascending=False).to_string(index=False))

# Strategy 3: RFE — 递归特征消除 (训练模型并迭代删除弱特征)
rfe = RFE(estimator=LinearRegression(), n_features_to_select=4).fit(X, y)
print('\nRFE Selected Features:')
for feat, sel in zip(X.columns, rfe.support_):
    print(f'  {feat:15s}: {"SELECTED" if sel else "removed"}')

## 4. Feature Scaling — StandardScaler vs MinMaxScaler

In [ ]:
sample = X.head(5)
print('Original (first 5 rows, 3 features):')
print(sample[['MedInc','HouseAge','Population']].round(3).to_string())

# StandardScaler: z = (x - mean) / std → mean=0, std=1
z = StandardScaler().fit_transform(sample[['MedInc','HouseAge','Population']])
print('\nAfter StandardScaler (mean=0, std=1):')
print(pd.DataFrame(z, columns=['MedInc','HouseAge','Population']).round(3).to_string())

# MinMaxScaler: z = (x - min) / (max - min) → range [0, 1]
mm = MinMaxScaler().fit_transform(sample[['MedInc','HouseAge','Population']])
print('\nAfter MinMaxScaler (range [0,1]):')
print(pd.DataFrame(mm, columns=['MedInc','HouseAge','Population']).round(3).to_string())

## 5. PCA — Dimensionality Reduction

In [ ]:
# PCA: reduce 8 dimensions → 2 while preserving as much variance as possible
X_scaled = StandardScaler().fit_transform(X)
pca = PCA().fit(X_scaled)
X_pca = pca.transform(X_scaled)

print('Explained Variance Ratio:')
for i, (ev, cev) in enumerate(zip(pca.explained_variance_ratio_,
    np.cumsum(pca.explained_variance_ratio_))):
    print(f'  PC{i+1}: {ev:.2%} (cumulative: {cev:.2%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1,9), pca.explained_variance_ratio_, color='#2e86de', alpha=0.7)
axes[0].set_xlabel('PC'); axes[0].set_ylabel('Variance Ratio'); axes[0].set_title('Scree Plot')
sc = axes[1].scatter(X_pca[:,0], X_pca[:,1], c=y, cmap='viridis', alpha=0.4, s=5)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('PCA 2D Projection')
plt.colorbar(sc, ax=axes[1], label='House Value')
plt.tight_layout(); plt.show()

## Check Your Understanding
1. IQR uses 1.5 as multiplier. What happens if you use 3.0 instead? (More/fewer outliers?)
2. What is the difference between StandardScaler and MinMaxScaler? When would you use each?
3. After PCA, how much variance is explained by the first 2 components? What is the trade-off of keeping only 2 dimensions?
4. If two features are correlated at r=0.95, what should you do? Drop one? Use PCA? Something else?